In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [60]:
# 1. 데이터 불러오기
iris = load_iris()
X = iris.data
y = iris.target

In [61]:
# 2. 표준화
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [62]:
# 3. 학습데이터와 테스트데이터 쪼개기
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [63]:
# numpy 배열 -> PyTorch 타입 배열(텐서)로 변환하자
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [64]:
# 모델한테 전달하기 위해서 데이터 셋을 만들고 DataLoader에게 전달해야 한다.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True) #
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) #

In [65]:
# 4. 모델정의 -> 클래스로 만든다. nn.Module 라는 클래스를 반드시 상속받아야 한다.
# 부모클래스가 하는 일이 많을 때 이런식으로 설계를 한다.
class IrisClassifier(nn.Module):
    def __init__(self): # 생성자
        super(IrisClassifier, self).__init__() # 부모 생성자를 호출한다.
                                               # 부모 생성자 호출코드는 메소드의 젤 처음에 와야한다.
                                               # super가 부모를 뜻함. 두개의 매개변수를 전달한다.
        # 입력은닉층 (iris는 4개의 특성을 갖는다.)
        self.fc1 = nn.Linear(4, 16) # fc1은 그냥 변수임 모델을 저장해둔다. nn.Linear(입력개수, 출력개수)
        self.fc2 = nn.Linear(16, 8)# 은닉층
        self.fc3 = nn.Linear(8, 3) # 출력층(출력결과가 3이어야 한다.) 파이토치는 softmax 함수 안쓴다.
                                    # 손실함수에서 softmax 함수가 호출된다.
        self.active = nn.ReLU() # 활성화 함수 - relu
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.active(x)
        x = self.fc2(x)
        x = self.active(x)
        x = self.fc3(x)
        return x

In [66]:
# 5. 모델 만들고, 손실함수, 옵티마이저
model = IrisClassifier()
criterion = nn.CrossEntropyLoss() # 손실함수 - 다중분류, 소프트맥스를 제공함
optimizer = optim.Adam(model.parameters(), lr=0.01) # 옵티마이저

In [67]:
# 6. 학습
def train_model(epochs):
    for epoch in range(epochs):
        # train_loader - 배치사이즈만큼
        for inputs, labels in train_loader: # 현재 배치사이즈 16개임, 16개씩 가져온다.
            optimizer.zero_grad() # 옵티마이저 초기화
            outputs = model(inputs) # 순전파, 가중치 계산중
            loss = criterion(outputs, labels) # 손실값을 계산한다.
            loss.backward() # 오차의 역전파
            optimizer.step()
        print(f"Epochs[{epoch+1}/{epochs}], Loss:{loss.item():.4f}")

    print("학습완료")

In [68]:
def evaluate_model():
    # 모델을 평가모드로 변경한다.
    model.eval()
    
    with torch.no_grad(): # 그라디언트 계산 비활성화
        correct_train = 0 # 훈련셋이 예측이 잘 맞는 경우 카운트하기 위한 변수
        total_train = 0 # 전체 개수
        for inputs, labels in train_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1) # 출력값이 확률, np.argmax 쓰듯이 젤 확률이 높은 찾기
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            # 현재 16개씩 처리하고 있음
        accuracy_train = 100 * correct_train/total_train
        print(f"훈련셋 정확도 : {accuracy_train}")

    with torch.no_grad(): # 그라디언트 계산 비활성화
        correct_test = 0 # 훈련셋이 예측이 잘 맞는 경우 카운트하기 위한 변수
        total_test = 0 # 전체 개수
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1) # 출력값이 확률, np.argmax 쓰듯이 젤 확률이 높은 찾기
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
            # 현재 16개씩 처리하고 있음
        accuracy_train = 100 * correct_test/total_test
        print(f"테스트셋 정확도 : {accuracy_train}")

In [69]:
if __name__ == "__main__":
    train_model(epochs=100)
    evaluate_model()

Epochs[1/100], Loss:0.9160
Epochs[2/100], Loss:0.4328
Epochs[3/100], Loss:0.6460
Epochs[4/100], Loss:0.4512
Epochs[5/100], Loss:0.4501
Epochs[6/100], Loss:0.2270
Epochs[7/100], Loss:0.2835
Epochs[8/100], Loss:0.1194
Epochs[9/100], Loss:0.1424
Epochs[10/100], Loss:0.2195
Epochs[11/100], Loss:0.1576
Epochs[12/100], Loss:0.2538
Epochs[13/100], Loss:0.2299
Epochs[14/100], Loss:0.0214
Epochs[15/100], Loss:0.0076
Epochs[16/100], Loss:0.2713
Epochs[17/100], Loss:0.0195
Epochs[18/100], Loss:0.0070
Epochs[19/100], Loss:0.1190
Epochs[20/100], Loss:0.0030
Epochs[21/100], Loss:0.0216
Epochs[22/100], Loss:0.0256
Epochs[23/100], Loss:0.0007
Epochs[24/100], Loss:0.0514
Epochs[25/100], Loss:0.0872
Epochs[26/100], Loss:0.0323
Epochs[27/100], Loss:0.0215
Epochs[28/100], Loss:0.0650
Epochs[29/100], Loss:0.0011
Epochs[30/100], Loss:0.0018
Epochs[31/100], Loss:0.0122
Epochs[32/100], Loss:0.0052
Epochs[33/100], Loss:0.0298
Epochs[34/100], Loss:0.3372
Epochs[35/100], Loss:0.0185
Epochs[36/100], Loss:0.2945
E